**Functions as First Class Citizens**

In [3]:
def greet(name):
  return f"Hello, {name}"
say_hello= greet
print(say_hello("John"))

# Pass as an argument
def execute_function(func, arg):
  return func(arg)
result = execute_function(greet, "Jane")
print(result)

# Return from function
def get_greeting_function():
  def greet_morning(name):
    return f"Good morning, {name}"
  return greet_morning

morning_greet = get_greeting_function()
print(morning_greet("Bob"))

Hello, John
Hello, Jane
Good morning, Bob


**Closures**

In [4]:
# Function that remembers
def make_multiplier(factor):
  def multiplier(number):
    return number * factor
  return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))
print(triple(5))

10
15


**Decorator**

In [7]:
""" A decorator is a function that takes another function and
    extends its behavior without modifying it.
"""
def my_decorator(func):
  def wrapper():
    print("Before function call")
    func()
    print("After function call")
  return wrapper

# Manual Decration
@my_decorator
def say_hello():
  print("Hello!")
say_hello()

# say_hello = my_decorator(say_hello)

Before function call
Hello!
After function call


**Decorator with Arguments**

In [9]:
import time
def timing_decorator(func):
  """Measure how long a function takes."""
  def wrapper(*args, **kwargs):
    start = time.time()
    result = func(*args, **kwargs)
    end = time.time()
    print(f"{func.__name__} took {end - start:.4f} seconds")
    return result
  return wrapper

@timing_decorator
def slow_function(n):
  """Stimulating slow operation"""
  time.sleep(n)
  return "Done"

result= slow_function(2)
print(result)

slow_function took 2.0001 seconds
Done


In [11]:
# Logging Decorator
import functools
from datetime import datetime

def log_calls(func):
    """Log every time a function is called."""

    @functools.wraps(func)  # Preserves function metadata
    def wrapper(*args, **kwargs):
        timestamp = datetime.now().isoformat()
        print(f"[{timestamp}] Calling {func.__name__} with args={args}, kwargs={kwargs}")

        try:
            result = func(*args, **kwargs)
            print(f"[{timestamp}] {func.__name__} returned {result}")
            return result
        except Exception as e:
            print(f"[{timestamp}] {func.__name__} raised {type(e).__name__}: {e}")
            raise

    return wrapper

@log_calls
def divide(a, b):
    """Divide two numbers."""
    return a / b

result = divide(10, 2)
divide(10, 2)

[2026-09-14T14:20:41.439535] Calling divide with args=(10, 2), kwargs={}
[2026-09-14T14:20:41.439535] divide returned 5.0
[2026-09-14T14:20:41.439669] Calling divide with args=(10, 2), kwargs={}
[2026-09-14T14:20:41.439669] divide returned 5.0


5.0

**Deorator With Parameters**

In [12]:
def repeat(times):
    """Decorator that repeats a function call."""

    def decorator(func):
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                result = func(*args, **kwargs)
                results.append(result)
            return results
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    return f"Hello, {name}!"

result = greet("John")
print(result)

['Hello, John!', 'Hello, John!', 'Hello, John!']


In [13]:
# Rate Limiting
import time
from functools import wraps

def rate_limit(max_calls, period):
    """Limit function calls to max_calls per period (seconds)."""

    def decorator(func):
        calls = []

        @wraps(func)
        def wrapper(*args, **kwargs):
            now = time.time()

            # Remove old calls outside the time window
            calls[:] = [call_time for call_time in calls if now - call_time < period]

            if len(calls) >= max_calls:
                raise Exception(f"Rate limit exceeded: {max_calls} calls per {period} seconds")

            calls.append(now)
            return func(*args, **kwargs)

        return wrapper
    return decorator

@rate_limit(max_calls=3, period=10)  # 3 calls per 10 seconds
def api_call():
    print("API called")
    return "Success"

# First 3 calls work
api_call()  # ✓
api_call()  # ✓
api_call()  # ✓

# 4th call raises exception
try:
    api_call()  # ✗ Rate limit exceeded
except Exception as e:
    print(f"Error: {e}")

API called
API called
API called
Error: Rate limit exceeded: 3 calls per 10 seconds


In [14]:
# Class Decorators
class CountCalls:
    """Decorator that counts function calls."""

    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"{self.func.__name__} has been called {self.count} times")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello!")

say_hello()  # say_hello has been called 1 times
say_hello()  # say_hello has been called 2 times
say_hello()  # say_hello has been called 3 times

say_hello has been called 1 times
Hello!
say_hello has been called 2 times
Hello!
say_hello has been called 3 times
Hello!


**Authentication Decorator**

In [17]:
from functools import wraps
def login_reqd(func):
  """Req user to be logged in"""
  @wraps(func)
  def wrapper(*args, **kwargs):
    user = kwargs.get('user')

    if user is None:
       raise PermissionError("Authentication required")
    if not user.get('is_authenticated'):
        raise PermissionError("User not authenticated")

    return func(*args, **kwargs)
  return wrapper

def admin_required(func):
    """Require user to be admin."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        user = kwargs.get('user')

        if not user.get('is_admin'):
            raise PermissionError("Admin access required")

        return func(*args, **kwargs)
    return wrapper

@login_reqd
def view_dashboard(user=None):
  return f"Dashboard for {user['name']}"

@login_reqd
@admin_required
def delete_user(user_id, user=None):
    return f"User {user_id} deleted by {user['name']}"

# Test
regular_user = {"name": "John", "is_authenticated": True, "is_admin": False}
admin_user = {"name": "Admin", "is_authenticated": True, "is_admin": True}
print(view_dashboard(user=regular_user))

try:
    delete_user(123, user=regular_user)
except PermissionError as e:
    print(f"Error: {e}")

print(delete_user(123, user=admin_user))


Dashboard for John


**Caching(Memorization)**

In [18]:
from functools import wraps

def cache(func):
    """Cache function results."""

    cached_results = {}

    @wraps(func)
    def wrapper(*args):
        if args in cached_results:
            print(f"Cache hit for {args}")
            return cached_results[args]

        print(f"Cache miss for {args}, computing...")
        result = func(*args)
        cached_results[args] = result
        return result

    return wrapper

@cache
def fibonacci(n):
    """Compute fibonacci number (slow without cache)."""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(10))  # Computes once
print(fibonacci(10))

Cache miss for (10,), computing...
Cache miss for (9,), computing...
Cache miss for (8,), computing...
Cache miss for (7,), computing...
Cache miss for (6,), computing...
Cache miss for (5,), computing...
Cache miss for (4,), computing...
Cache miss for (3,), computing...
Cache miss for (2,), computing...
Cache miss for (1,), computing...
Cache miss for (0,), computing...
Cache hit for (1,)
Cache hit for (2,)
Cache hit for (3,)
Cache hit for (4,)
Cache hit for (5,)
Cache hit for (6,)
Cache hit for (7,)
Cache hit for (8,)
55
Cache hit for (10,)
55


In [19]:
# Built-in cache
from functools import lru_cache

@lru_cache(maxsize=128)  # Least Recently Used cache
def expensive_computation(x, y):
    print(f"Computing {x} + {y}")
    return x + y

expensive_computation(1, 2)  # Computing 1 + 2
expensive_computation(1, 2)  # Returns from cache (no print)

Computing 1 + 2


3

**Retry Decorator**

In [20]:
import time
from functools import wraps

def retry(max_attempts=3, delay=1):
    """Retry a function if it fails."""

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0

            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts >= max_attempts:
                        raise
                    print(f"Attempt {attempts} failed: {e}. Retrying in {delay}s...")
                    time.sleep(delay)

        return wrapper
    return decorator

@retry(max_attempts=3, delay=2)
def unreliable_api_call():
    """Simulate an unreliable API."""
    import random
    if random.random() < 0.7:  # 70% chance of failure
        raise ConnectionError("API unavailable")
    return "Success"

result = unreliable_api_call()
print(result)

Attempt 1 failed: API unavailable. Retrying in 2s...
Success
